# Intelligent Loan Risk Assessment — Analysis Notebook
This notebook mirrors the production pipeline in `ml/` for documentation, exploration, and viva walkthrough purposes. Run cells top to bottom.

## 1. Load Data

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../ml'))
import pandas as pd
import numpy as np

df = pd.read_csv('../dataset/loan_dataset.csv')
df.shape

In [ ]:
df.head()

## 2. Exploratory Data Analysis

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
df.duplicated().sum()

In [ ]:
df['risk_category'].value_counts(normalize=True)

In [ ]:
df.describe().T

## 3. Cleaning + Feature Engineering
Uses the same `preprocessing.py` / `feature_engineering.py` modules as the production pipeline, so this notebook can never drift from what the Flask backend actually does.

In [ ]:
from preprocessing import clean_raw_data, build_preprocessor
from feature_engineering import engineer_features, select_model_columns

df_clean = clean_raw_data(df)
df_feat = engineer_features(df_clean)
df_feat[['debt_to_income_ratio','loan_to_income_ratio','monthly_repayment_estimate','income_stability_score']].describe()

## 4. Train/Test Split + Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

y = df_feat['risk_category']
X = select_model_columns(df_feat)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = build_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
if hasattr(X_train_proc, 'toarray'):
    X_train_proc = X_train_proc.toarray(); X_test_proc = X_test_proc.toarray()
X_train_proc.shape

## 5. SMOTE Balancing (training set only)

In [ ]:
from smote_custom import SMOTE

print('Before:', pd.Series(y_train).value_counts().to_dict())
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_proc, y_train.to_numpy())
print('After:', pd.Series(y_train_bal).value_counts().to_dict())

## 6. Train & Compare Models
See `ml/train_models.py` for the full training + tuning + artifact-saving pipeline. This section reproduces a quick comparison inline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelBinarizer

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost (GradientBoosting)': GradientBoostingClassifier(random_state=42),
}

lb = LabelBinarizer().fit(['Low','Medium','High'])
results = []
for name, m in models.items():
    m.fit(X_train_bal, y_train_bal)
    pred = m.predict(X_test_proc)
    proba = m.predict_proba(X_test_proc)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='weighted')
    try:
        auc = roc_auc_score(lb.transform(y_test), proba, average='weighted', multi_class='ovr')
    except ValueError:
        auc = float('nan')
    results.append({'model': name, 'accuracy': round(acc,4), 'f1_score': round(f1,4), 'roc_auc': round(auc,4)})

pd.DataFrame(results).sort_values('roc_auc', ascending=False)

## 7. Load the Production Model
The actual saved model used by the Flask backend (trained by `ml/train_models.py`, including hyperparameter tuning).

In [ ]:
import joblib, json

model = joblib.load('../ml/model.pkl')
with open('../ml/model_meta.json') as f:
    meta = json.load(f)
meta

## 8. Feature Importance (Explainability)

In [ ]:
with open('../ml/feature_importance.json') as f:
    fi = json.load(f)
pd.DataFrame(fi)

## 9. Model Comparison Table (from saved training run)

In [ ]:
pd.read_csv('../ml/model_comparison.csv')

## 10. Conclusion
The saved pipeline (`preprocessor.pkl` + `model.pkl`) is loaded directly by the Flask backend (`backend/utils/predictor.py`) at request time, guaranteeing the exact same feature engineering and preprocessing seen here is applied to every live prediction.